In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-25 07:17:04--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  4.61MB/s    in 0.2s    

2026-03-25 07:17:05 (4.61 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print('length of dataset in characters: ', len(text))

length of dataset in characters:  1115394


In [4]:
# let's looks at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# Building Vocab
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a stringArithmeticError

# This is a character level tokenizer
# Google uses sentencepiece tokenizer.
# OpenAI uses tiktoken 

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# let's now encode the enitre text dataset and store it into a torch.Tensor
import torch # we use PyTorch
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
# print(data[:1000]) # the first 1000 characters in numerical tensor form

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) torch.int64


In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
print(train_data.shape)
type(train_data)

torch.Size([1003854])


torch.Tensor

In [10]:
# Context Window
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target: {target}')

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # maximum context length for predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # We grab 4 (batch_size) random index values from the corpus and create the batch from them.
    # So suppose we find the index n, we get the first training example from n to n+8 (block_size=8)
    # And we create a +1 offset for the target.

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension (num_tokens)
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target : {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target : 43
when input is [24, 43] the target : 58
when input is [24, 43, 58] the target : 5
when input is [24, 43, 58, 5] the target : 57
when input is [24, 43, 58, 5, 57] the target : 1
when input is [24, 43, 58, 5, 57, 1] the target : 46
when input is [24, 43, 58, 5, 57, 1, 46] the target : 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target : 39
when input is [44] the target : 53
when input is [44, 53] the target : 56
when input is [44, 53, 56] the target : 1
when input is [44, 53, 56, 1] the target : 58
when input is [44, 53, 56, 1, 58] the target : 46
when inp

In [13]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [14]:
import blocks
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [15]:
class GPT(nn.Module):
    """ One layer of GPT with attention and FFNN """

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.attention = blocks.MaskedMultiHeadAttention(d_model, num_heads)
        self.ffnn = blocks.FFNN(d_model, hidden_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of x: (Batch_size, num_tokens, d_model)
        x = x + self.dropout(self.attention(self.norm1(x))) # Residual connection with Pre Normalization
        x = x + self.dropout(self.ffnn(self.norm2(x))) # Residual connection with Pre Normalization
        # Dimension: (Batch_size, num_tokens, d_model)
        return x

In [30]:
class GPTStack(nn.Module):
    """ A stack of GPT Layers """

    def __init__(self, vocab_size: int, d_model: int, num_blocks: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model     
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = blocks.PositionalEncoding()
        self.layers = nn.ModuleList([GPT(d_model, num_heads, hidden_dim, dropout_rate) for _ in range(num_blocks)])
        self.final_output_logits = nn.Linear(d_model, vocab_size)
        self.norm1 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, targets: torch.Tensor = None) -> torch.Tensor:
        # Dimension of tokens: (Batch_size, num_tokens)

        x = self.embedding(x)
        x = self.positional_encoding(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        for layer in self.layers:
            x = layer(x)
        
        norm_output = self.norm1(x)

        logits = self.final_output_logits(norm_output)
        # Dimension: (Batch_size, num_tokens, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # Reshaping our Tensors
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, x: torch.Tensor, max_new_tokens: int) -> torch.Tensor:

        for _ in range(max_new_tokens):

            logits, loss = self(x)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            x_next = torch.multinomial(probs, num_samples=1)

            x = torch.cat((x, x_next), dim=1)

        return x
        

In [31]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from the lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)
        # Dimension: (Batch, Time, Embedding)

        # Now, the cross_entropy loss function in pytorch expects the Channel Dimension (Embedding Dimension) 
        # to be at the 2nd dimension in the tensor. 
        # Therefore, we will reshape our logits to merge the Batch and Time dimension to bring C to the 2nd dimension 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # Reshaping our Tensors
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):

            # get the predictions
            logits, loss = self(idx) # We are calling the forward function of the model
            # logits Dimension: (Batch_size, Time, Logits)

            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # Dimension: (Batch_size, Logits)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Picking the index of with probability directly proportional to the value at the index
            # Dimension: (Batch_size, 1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            
        return idx

In [32]:
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 16 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
d_model = 384
hidden_dim = d_model * 4
n_head = 6
n_layer = 6
dropout = 0.2

In [33]:
model = GPTStack(vocab_size, d_model, n_layer, n_head, hidden_dim, dropout)
model

GPTStack(
  (embedding): Embedding(65, 384)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-5): 6 x GPT(
      (attention): MaskedMultiHeadAttention(
        (w_q): Linear(in_features=384, out_features=384, bias=False)
        (w_k): Linear(in_features=384, out_features=384, bias=False)
        (w_v): Linear(in_features=384, out_features=384, bias=False)
        (w_o): Linear(in_features=384, out_features=384, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=384, out_features=1536, bias=True)
        (output_layer): Linear(in_features=1536, out_features=384, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (final_output_logits): Linear(in_features=384, out_features=65, bias=True)
  (

In [34]:
logits, loss = model(xb, yb)
print(logits.shape)
print(f'loss: {loss.item()}')

torch.Size([32, 65])
loss: 4.526690483093262


In [35]:
idx = torch.zeros((1, 1), dtype=torch.long)
# Creating an initial input with dimension: (Batch_Size, Time, Num_Tokens) 

output = model.generate(idx, max_new_tokens=100)
# Calling generate function
print(output.shape)

print(decode(output[0].tolist()))
# printing the first 

torch.Size([1, 101])

BWSH
?w
JroJaC3!WPyc'Ruyrgnvdmf?LbSTBy-f
mTA!,$WMdU
.LfBSH.rQOyp3F qDRt?KI
Q
BUS LARVtT.qD;qAc!WTlOj


In [37]:
output = model.generate(x = torch.zeros((4, 1), dtype=torch.long), max_new_tokens=100)
# Passing batch_size as 4 and getting 4 batch's predictions in parallel
output.shape

torch.Size([4, 101])

In [38]:
# Decoding all 4 batches
for i in range(4):
    print(decode(output[i].tolist()))



?droNcARjDDO&E.:uR!Cm OQOKPs?DO&dpBBToRgNFySFwPK,R&-$UYHHSkqEklJ
uVvHRFfgSM3?SFdL
j
 zKqZDWDxTp,oCX;

wN.NJa-SRjeOgMitBDxhoAS-ZaEqRqCOBz'z!dL'IDa,XrfDkFI$xJEhSsvv?'WSSSXlRi-GLpFJ-mSHTE,K.;uSLp'vu3;!-nOQ

QmJEzj?c:NaYmxtgiZmAR:kow &dARg
gZrdDtiSfpWgl$S!,DiqjNuAROcxKSpNsZp
XE:gsGdB'vBQ;v:STlscW YHZKwHnuVA

:XO3SQ$FE
jChlqkjXBDNV$tjLpcomash
mY$OSOCGk'lMzzajcSu-ZxLJihVu,QuuC'CZ!'!,Ta'jt.zZTA3WREO,
jn'IExiHk


In [48]:
# Create a Pytorch Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 3e-05
    maximize: False
    weight_decay: 0.01
)

In [49]:
batch_size=32
for step in range(1000):

    # sample a batch of data from our dataloader
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()

    print(loss.item())
    

2.030731678009033
1.9578877687454224
2.024904727935791
2.017836332321167
1.9834802150726318
2.014525890350342
2.054492712020874
1.9054399728775024
1.9069890975952148
2.0070383548736572
1.9555052518844604
2.039149761199951
2.095548629760742
1.9433555603027344
2.0056591033935547
2.0664544105529785
2.0939104557037354
1.987949013710022
1.9977309703826904
1.927547812461853
1.983944296836853
2.106471300125122
1.9879872798919678
1.9645094871520996
1.95985746383667
1.9475356340408325
2.0509588718414307
2.0462100505828857
2.0256078243255615
2.0666167736053467
2.0725600719451904
1.9993816614151
1.9884963035583496
2.0986077785491943
1.9303840398788452
1.9766324758529663
1.9202226400375366
1.937911868095398
2.065980911254883
1.958608627319336
1.840411901473999
1.925115942955017
1.9816811084747314
1.8960928916931152
1.9540882110595703


KeyboardInterrupt: 

In [50]:
print(loss.item())


1.9540882110595703


In [51]:
idx = torch.zeros((1, 1), dtype=torch.long)
# Creating an initial input with dimension: (Batch_Size, Time, Num_Tokens) 

output = model.generate(idx, max_new_tokens=400)
# Calling generate function
print(output.shape)

print(decode(output[0].tolist()))
# printing the first 

torch.Size([1, 401])

CUSalve pread:
Ecoorieeanoourte proureiselvirareRorore r,nerep, hayorencerplldowasp dialsie woniase:
I hiound ayowh co blyortrn h wist blltengr juthoablln iouchorveyerditraveivendothasendDehar siatharndsiurnyaron has, blleyothn courcheemoryearsterrestitind oun tinsiveatyo, rd win'dourtstouchad Bersp y; s d p och ghthenctiveyorveers teryooulawn arenurervindThe trvalverncteacinos Pwen mereren urvrk,
